# Methodes proposees

Execute B1, B2, B3 et leur combinaison.

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().resolve()
while project_root != project_root.parent:
    if (project_root / 'src').is_dir() and (project_root / 'mvtec_anomaly_detection').is_dir():
        break
    project_root = project_root.parent
else:
    raise RuntimeError('Racine du projet LightShift-PatchCore introuvable')

%cd {project_root}
sys.path.insert(0, str(project_root))

In [ ]:
import subprocess
import sys

methods = [
    ('baseline', []),
    ('b1', ['--b1']),
    ('b2', ['--b2']),
    ('b3', ['--b3']),
    ('b1_b2_b3', ['--b1', '--b2', '--b3'])
]

for name, flags in methods:
    output_dir = 'artifacts' if name == 'baseline' else 'artifacts_' + name
    subprocess.run([
        sys.executable,
        '-m', 'src.experiments.run_experiments',
        '--root', 'mvtec_anomaly_detection',
        '--categories', 'bottle',
        '--lighting', 'dim',
        '--batch-size', '4',
        '--output', output_dir,
        *flags
    ], cwd=project_root, check=True)

In [ ]:
import glob
import json
import pandas as pd

rows = []
roots = ['artifacts', 'artifacts_b1', 'artifacts_b2', 'artifacts_b3', 'artifacts_b1_b2_b3']
for root in roots:
    for path in glob.glob(str(project_root / root / 'bottle/*/metrics.json')):
        with open(path, encoding='utf-8') as handle:
            row = json.load(handle)
            row['method'] = 'baseline' if root == 'artifacts' else root.replace('artifacts_', '')
            rows.append(row)

pd.DataFrame(rows)[['method', 'lighting', 'image_auroc', 'pixel_auroc', 'accuracy', 'threshold']]